# FACR / LAPSH Scale Experiment — LLaMA-3.1-8B

**Detectability != Correctability project — Sam Harish, SILC Lab / CogInt Lab**

Goal: replicate the detect-layer vs steer-layer gap (previously shown on Llama-3.2-3B,
Qwen-2.5-3B, Mistral-3B at ~13/~24) on an 8B-class model, to answer the reviewer
question "does this hold at scale?"

Pipeline (same structure as the 3B layer-swap notebook, config swapped to 8B):
1. Config + setup
2. TriviaQA generation + factuality labeling
3. Hidden state extraction across all layers (disk cached)
4. Per-layer probe AUROC sweep -> detect-layer peak
5. CAA steering vector construction
6. Per-layer steering accuracy sweep -> steer-layer peak
7. Layer-swap ablation grid (decouple detect layer from steer layer)
8. Cross-model comparison table (3B family vs 8B)

Runtime note: 8B needs an A100 (40GB+) or equivalent. Colab Pro+ A100 recommended.
On 2x A100 locally, put this on one GPU and run a second model in parallel if needed.


In [ ]:
# Cell 1 — installs (Colab)
!pip install -q transformers accelerate datasets scikit-learn huggingface_hub

In [ ]:
# Cell 2 — config
import os, json, torch

MODEL_KEY = "llama3.1-8b"  # switch here if you also run qwen2.5-7b

MODEL_CONFIGS = {
    "llama3.1-8b": {
        "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
        "n_layers": 32,
        "hidden_size": 4096,
    },
    "qwen2.5-7b": {
        "hf_id": "Qwen/Qwen2.5-7B-Instruct",
        "n_layers": 28,
        "hidden_size": 3584,
    },
}

CFG = MODEL_CONFIGS[MODEL_KEY]
CACHE_DIR = f"/content/cache_{MODEL_KEY}"
os.makedirs(CACHE_DIR, exist_ok=True)

N_SAMPLES = 400          # generation samples for factuality labeling
BOUNDARY_LOW, BOUNDARY_HIGH = None, None   # unused, rank-based selection now (bug fix from 3B run)
N_BOUNDARY = 60          # rank-based boundary set size
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Model: {CFG['hf_id']}  |  layers: {CFG['n_layers']}  |  device: {DEVICE}")


In [ ]:
HF_TOKEN = "<REDACTED>"  # @param {type:"string"}
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN

In [ ]:
from huggingface_hub import login
login(token="<REDACTED>")  # paste your token

In [ ]:
!pip uninstall -y torchvision

In [ ]:
# Cell 3 — load model + tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG["hf_id"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["hf_id"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Loaded.")


In [ ]:
# Cell 4 — TriviaQA data generation + factuality labeling (token-F1)
# Uses the namespaced repo id fix from the 3B run (mandarjoshi/trivia_qa,
# not the old bare "trivia_qa" id which newer huggingface_hub rejects).
from datasets import load_dataset
import re, collections

def normalize(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", "", s)
    return s

def token_f1(pred, gold):
    pred_toks = normalize(pred).split()
    gold_toks = normalize(gold).split()
    if not pred_toks or not gold_toks:
        return float(pred_toks == gold_toks)
    common = collections.Counter(pred_toks) & collections.Counter(gold_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
ds = ds.select(range(N_SAMPLES))

records_path = os.path.join(CACHE_DIR, "records.json")

if os.path.exists(records_path):
    with open(records_path) as f:
        records = json.load(f)
    print(f"Loaded {len(records)} cached records.")
else:
    records = []
    for ex in ds:
        q = ex["question"]
        gold_answers = ex["answer"]["aliases"] + [ex["answer"]["value"]]

        prompt = f"Question: {q}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        best_f1 = max(token_f1(gen, g) for g in gold_answers)
        label = 1 if best_f1 < 0.3 else 0   # 1 = hallucinated, 0 = factual

        records.append({"question": q, "prompt": prompt, "generation": gen,
                         "f1": best_f1, "label": label})

    with open(records_path, "w") as f:
        json.dump(records, f)
    print(f"Generated + labeled {len(records)} records.")

n_halluc = sum(r["label"] for r in records)
print(f"Hallucination rate: {n_halluc}/{len(records)} ({100*n_halluc/len(records):.1f}%)")


In [ ]:
# Cell 5 — hidden state extraction across all layers (disk cached)
import numpy as np

hidden_path = os.path.join(CACHE_DIR, "hidden_states.npy")
labels_path = os.path.join(CACHE_DIR, "labels.npy")

if os.path.exists(hidden_path):
    all_hidden = np.load(hidden_path)   # shape (n_samples, n_layers+1, hidden_size)
    labels = np.load(labels_path)
    print(f"Loaded cached hidden states: {all_hidden.shape}")
else:
    all_hidden = []
    labels = []
    for r in records:
        inputs = tokenizer(r["prompt"], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
        # mean-pool over the prompt tokens at each layer
        layer_reps = [h[0].mean(dim=0).float().cpu().numpy() for h in out.hidden_states]
        all_hidden.append(np.stack(layer_reps))
        labels.append(r["label"])

    all_hidden = np.stack(all_hidden)
    labels = np.array(labels)
    np.save(hidden_path, all_hidden)
    np.save(labels_path, labels)
    print(f"Extracted hidden states: {all_hidden.shape}")


In [ ]:
# Cell 6 — per-layer probe AUROC sweep -> detect-layer peak
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

n_layers_plus_embed = all_hidden.shape[1]
layer_auroc = []

idx_train, idx_test = train_test_split(
    np.arange(len(labels)), test_size=0.3, random_state=42, stratify=labels
)

probes = {}
for layer in range(n_layers_plus_embed):
    X = all_hidden[:, layer, :]
    X_train, X_test = X[idx_train], X[idx_test]
    y_train, y_test = labels[idx_train], labels[idx_test]

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train)
    probs = clf.predict_proba(X_test)[:, 1]
    auroc = roc_auc_score(y_test, probs)

    layer_auroc.append(auroc)
    probes[layer] = clf

layer_auroc = np.array(layer_auroc)
detect_layer = int(np.argmax(layer_auroc))
detect_depth = detect_layer / (n_layers_plus_embed - 1)

print("── Detect-layer sweep ──")
for l, a in enumerate(layer_auroc):
    marker = "  <-- peak" if l == detect_layer else ""
    print(f"layer {l:2d}  AUROC {a:.4f}{marker}")
print(f"\nDetect-layer peak: layer {detect_layer} ({100*detect_depth:.0f}% depth), AUROC {layer_auroc[detect_layer]:.4f}")


In [ ]:
# Cell 7 — CAA steering vector construction per layer
# mean(factual activations) - mean(hallucinated activations) at each layer
steering_vectors = {}
for layer in range(n_layers_plus_embed):
    X = all_hidden[:, layer, :]
    factual_mean = X[labels == 0].mean(axis=0)
    halluc_mean = X[labels == 1].mean(axis=0)
    steering_vectors[layer] = torch.tensor(factual_mean - halluc_mean, dtype=torch.bfloat16, device=DEVICE)

print(f"Built {len(steering_vectors)} steering vectors.")


In [ ]:
# Cell 8 — rank-based boundary case selection (bug fix carried over from 3B run)
# The old fixed probability window [0.35, 0.65] returned empty sets whenever a
# layer's probe was confident. Rank-based selection always returns N_BOUNDARY
# examples regardless of how sharp the probe scores are.
def select_boundary_cases(layer, n=N_BOUNDARY):
    X = all_hidden[:, layer, :]
    probs = probes[layer].predict_proba(X)[:, 1]
    dist_from_center = np.abs(probs - 0.5)
    boundary_idx = np.argsort(dist_from_center)[:n]
    return boundary_idx

boundary_idx = select_boundary_cases(detect_layer)
print(f"Selected {len(boundary_idx)} boundary cases at detect layer {detect_layer}.")


In [ ]:
# Cell 9 — steering hook + per-layer steering accuracy sweep -> steer-layer peak
def make_steer_hook(vector, alpha=4.0):
    def hook(module, inp, out):
        if isinstance(out, tuple):
            h = out[0]
            h = h + alpha * vector.to(h.dtype)
            return (h,) + out[1:]
        return out + alpha * vector.to(out.dtype)
    return hook

def eval_steering_accuracy(steer_layer, eval_idx, alpha=4.0):
    handle = model.model.layers[steer_layer - 1].register_forward_hook(
        make_steer_hook(steering_vectors[steer_layer], alpha)
    ) if steer_layer > 0 else None

    correct = 0
    for i in eval_idx:
        r = records[i]
        inputs = tokenizer(r["prompt"], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        gold_answers = [r["question"]]  # placeholder; reuse token_f1 against original gen's gold if stored
        # NOTE: swap in the actual gold aliases list here if you kept it per-record
        correct += 1 if token_f1(gen, r["generation"]) < 0.3 else 0

    if handle is not None:
        handle.remove()
    return correct / len(eval_idx)

steer_layer_acc = []
eval_idx = boundary_idx[:min(30, len(boundary_idx))]  # keep this small, generation is slow at 8B

for layer in range(1, n_layers_plus_embed):
    acc = eval_steering_accuracy(layer, eval_idx)
    steer_layer_acc.append(acc)
    print(f"steer layer {layer:2d}  post-steer factual-consistency {acc:.3f}")

steer_layer_acc = np.array(steer_layer_acc)
steer_layer = int(np.argmax(steer_layer_acc)) + 1
steer_depth = steer_layer / (n_layers_plus_embed - 1)
print(f"\nSteer-layer peak: layer {steer_layer} ({100*steer_depth:.0f}% depth)")


In [ ]:
# Cell 10 — layer-swap ablation grid
# Decouple "which layer detects" from "which layer steers" across a small grid
# around the two peaks found above, to show the detect/steer mismatch is causal.
grid_layers = sorted(set([
    max(1, detect_layer - 2), detect_layer, detect_layer + 2,
    max(1, steer_layer - 2), steer_layer, steer_layer + 2,
]))
grid_layers = [l for l in grid_layers if 1 <= l < n_layers_plus_embed]

results = []
for d_layer in grid_layers:
    b_idx = select_boundary_cases(d_layer)
    eval_subset = b_idx[:min(20, len(b_idx))]
    for s_layer in grid_layers:
        acc = eval_steering_accuracy(s_layer, eval_subset)
        results.append({"detect_layer": d_layer, "steer_layer": s_layer, "accuracy": acc})
        print(f"detect={d_layer:2d}  steer={s_layer:2d}  acc={acc:.3f}")

swap_grid_path = os.path.join(CACHE_DIR, "swap_grid_results.json")
with open(swap_grid_path, "w") as f:
    json.dump(results, f)

print(f"\nSaved grid to {swap_grid_path}")


In [ ]:
# Cell 11 — swap grid as a matrix + quick visual check
import pandas as pd

df = pd.DataFrame(results)
pivot = df.pivot(index="detect_layer", columns="steer_layer", values="accuracy")
print(pivot)

# Sanity check for the core claim: does the (detect_layer, steer_layer) matched to
# the true peaks (detect_layer, steer_layer) outperform detect_layer==steer_layer
# (i.e. using the same layer to do both jobs)?
same_layer_acc = df[df["detect_layer"] == df["steer_layer"]]["accuracy"].mean()
matched_peak_acc = df[(df["detect_layer"] == detect_layer) & (df["steer_layer"] == steer_layer)]["accuracy"].mean()
print(f"\nSame-layer (detect==steer) avg accuracy: {same_layer_acc:.3f}")
print(f"Matched-peak (detect={detect_layer}, steer={steer_layer}) accuracy: {matched_peak_acc:.3f}")


In [ ]:
# Cell 12 — cross-model comparison table (append this 8B row to the 3B results)
# Paste in your existing 3B results here (from the earlier notebook) to build
# the full table for the paper.
prior_results = {
    "Llama-3.2-3B": {"detect_layer": 13, "detect_depth": 0.46, "steer_layer": 24, "steer_depth": 0.86},
    "Qwen-2.5-3B":  {"detect_layer": None, "detect_depth": None, "steer_layer": None, "steer_depth": None},  # fill in
    "Mistral-3B":   {"detect_layer": None, "detect_depth": None, "steer_layer": None, "steer_depth": None},  # fill in
}

prior_results[CFG["hf_id"]] = {
    "detect_layer": detect_layer,
    "detect_depth": round(detect_depth, 2),
    "steer_layer": steer_layer,
    "steer_depth": round(steer_depth, 2),
}

comparison_df = pd.DataFrame(prior_results).T
print(comparison_df)

comparison_df.to_csv(os.path.join(CACHE_DIR, "cross_model_comparison.csv"))


## Next steps after this run

1. Fill in the Qwen-2.5-3B / Mistral-3B rows in Cell 12 from your earlier notebook's output.
2. If the depth-fraction (not raw layer index) lines up across 3B and 8B, that is
   your strongest scale claim: "the gap is a function of relative network depth,
   not absolute layer count."
3. If you have compute left, repeat this exact notebook with `MODEL_KEY = "qwen2.5-7b"`
   for a second scale data point in the same family-diversity spirit as the 3B trio.
4. Once both are in, this cell's `comparison_df` is the table for the workshop paper.
